# Caderno 4 - Indexa usando BM25

## 1. Parâmetros

In [1]:
PASTA_DOCS_QRELS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/'
PASTA_RESULTADO_CADERNO = './dados/outputs/4 - indexacao bm25/'
PASTA_QRELS_LLMS = './dados/outputs/3 - qrel llm/'

ARQUIVO_DOCS = f'{PASTA_DOCS_QRELS}/docs.csv'
ARQUIVO_QUERIES = f'{PASTA_DOCS_QRELS}/query.csv'
ARQUIVO_QRELS = f'{PASTA_DOCS_QRELS}/qrel.csv'

NOME_ARQUIVO_INDICE_TEXTO = f'{PASTA_RESULTADO_CADERNO}indice_bm25_texto.pickle'
NOME_ARQUIVO_INDICE_TEXTO_E_ASSUNTO = f'{PASTA_RESULTADO_CADERNO}indice_bm25_texto_e_assunto.pickle'

## 2. Carrega os documentos, qrels e queries

In [2]:
import pandas as pd

docs = pd.read_csv(ARQUIVO_DOCS)
qrels = pd.read_csv(ARQUIVO_QRELS)
queries = pd.read_csv(ARQUIVO_QUERIES)

print('Total de NA nos campos assunto e texto da norma:')
print(docs['ASSUNTO'].isna().sum())
print(docs['TEXTONORMA'].isna().sum())

docs['ASSUNTO'] = docs['ASSUNTO'].fillna('')
docs['TEXTONORMA'] = docs['TEXTONORMA'].fillna('')

Total de NA nos campos assunto e texto da norma:
7190
5


## 3. Instancia BM25 e cria 2 índices: apenas o texto da norma / apenas o texto da norma e assunto

In [3]:
from bm25 import IndiceInvertido, BM25, tokenizador_pt_remove_html

import os

# Vamos criar um índice invertido e indexar o texto principal.
iidx_texto = IndiceInvertido(tokenizador_pt_remove_html)
iidx_texto_e_assunto = IndiceInvertido(tokenizador_pt_remove_html)

if not os.path.exists(NOME_ARQUIVO_INDICE_TEXTO):
    # Se for indexar a primeira vez (cerca de 1h):
    iidx_texto.adiciona_dataframe(docs, lambda row: (row['KEY'], row['TEXTONORMA']))
    iidx_texto.to_pickle(NOME_ARQUIVO_INDICE_TEXTO)
else:
    # Se quiser recuperar de um arquivo:
    iidx_texto.from_pickle(NOME_ARQUIVO_INDICE_TEXTO)

if not os.path.exists(NOME_ARQUIVO_INDICE_TEXTO_E_ASSUNTO):
    # Se for indexar a primeira vez (cerca de 1h):
    iidx_texto_e_assunto.adiciona_dataframe(docs, lambda row: (row['KEY'], row['ASSUNTO'] + ' ' + row['TEXTONORMA']))
    iidx_texto_e_assunto.to_pickle(NOME_ARQUIVO_INDICE_TEXTO_E_ASSUNTO)
else:
    # Se quiser recuperar de um arquivo:
    iidx_texto_e_assunto.from_pickle(NOME_ARQUIVO_INDICE_TEXTO_E_ASSUNTO)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\P_8454\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\P_8454\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\P_8454\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!


## 4. Pesquisa nos índices

In [4]:
# Agora instancia um BM25
buscador_texto_default_pyserini = BM25(iidx_texto, k1=0.9, b=0.4, bias_idf=1)

col_resultado_query_key=[]
col_resultado_doc_key=[]
col_resultado_rank=[]

for i, row in queries.iterrows():
    query_key = row.KEY
    query_text = row.TEXT
    resultados = buscador_texto_default_pyserini.pesquisar(query_text)

    primeiros_50_docs = [tupla_key_score[0] for tupla_key_score in resultados[:50]]
    queries_keys = [query_key] * len(primeiros_50_docs)
    ranking = list(range(1, len(primeiros_50_docs)+1))

    col_resultado_query_key.extend(queries_keys)
    col_resultado_doc_key.extend(primeiros_50_docs)
    col_resultado_rank.extend(ranking)

df_resultados_texto_default_pyserini = pd.DataFrame({
    "QUERY_KEY": col_resultado_query_key,
    "DOC_KEY": col_resultado_doc_key,
    "RANK": col_resultado_rank,
})

In [5]:
# Agora instancia um BM25
buscador_texto = BM25(iidx_texto, k1=0.82, b=0.68, bias_idf=1)

col_resultado_query_key=[]
col_resultado_doc_key=[]
col_resultado_rank=[]

for i, row in queries.iterrows():
    query_key = row.KEY
    query_text = row.TEXT
    resultados = buscador_texto.pesquisar(query_text)

    primeiros_50_docs = [tupla_key_score[0] for tupla_key_score in resultados[:50]]
    queries_keys = [query_key] * len(primeiros_50_docs)
    ranking = list(range(1, len(primeiros_50_docs)+1))

    col_resultado_query_key.extend(queries_keys)
    col_resultado_doc_key.extend(primeiros_50_docs)
    col_resultado_rank.extend(ranking)

df_resultados_texto = pd.DataFrame({
    "QUERY_KEY": col_resultado_query_key,
    "DOC_KEY": col_resultado_doc_key,
    "RANK": col_resultado_rank,
})

In [6]:
# Agora instancia um BM25
buscador_texto_e_assunto = BM25(iidx_texto_e_assunto, k1=0.82, b=0.68, bias_idf=1)

col_resultado_query_key=[]
col_resultado_doc_key=[]
col_resultado_rank=[]

for i, row in queries.iterrows():
    query_key = row.KEY
    query_text = row.TEXT
    resultados = buscador_texto_e_assunto.pesquisar(query_text)

    primeiros_50_docs = [tupla_key_score[0] for tupla_key_score in resultados[:50]]
    queries_keys = [query_key] * len(primeiros_50_docs)
    ranking = list(range(1, len(primeiros_50_docs)+1))

    col_resultado_query_key.extend(queries_keys)
    col_resultado_doc_key.extend(primeiros_50_docs)
    col_resultado_rank.extend(ranking)

df_resultados_texto_e_assunto = pd.DataFrame({
    "QUERY_KEY": col_resultado_query_key,
    "DOC_KEY": col_resultado_doc_key,
    "RANK": col_resultado_rank,
})

## 5. Mostra métricas

In [7]:
# Carrega todos os qrels de LLMs
from pathlib import Path

PASTA_QRELS_LLMS = Path(PASTA_QRELS_LLMS)

qrels_llms = {}

# Percorre todos os arquivos .csv da pasta
for arquivo in PASTA_QRELS_LLMS.glob("*.csv"):
    # Nome do arquivo sem extensão
    nome_qrels = arquivo.stem
    
    # Lê o CSV
    df = pd.read_csv(arquivo)
    
    # Salva no dicionário
    qrels_llms[nome_qrels] = df

print(f"{len(qrels_llms)} arquivos carregados.")

6 arquivos carregados.


In [11]:
from metricas import metricas

df_metricas_qrels_texto_default_pyserini = metricas(df_resultados_texto_default_pyserini, qrels, aproximacao_trec_eval=True, k=[5,10])
print('nDCG@10')
print(f'qrel humano: {df_metricas_qrels_texto_default_pyserini['MRR@10'].mean()}')
for nome_qrels, qrel_llm in qrels_llms.items():
    df_metricas_qrels_llm_texto_default_pyserini = metricas(df_resultados_texto_default_pyserini, qrel_llm, aproximacao_trec_eval=True, k=[5,10])
    print(f'{nome_qrels}: {df_metricas_qrels_llm_texto_default_pyserini['nDCG@10'].mean()}')

nDCG@10
qrel humano: 0.49660973084886134
qrel_deepseek-chat_cot: 0.30940723952568944
qrel_deepseek-chat_simple: 0.3246749720147716
qrel_gpt-5-mini-2025-08-07_cot: 0.2923168845105116
qrel_gpt-5-mini-2025-08-07_simple: 0.2997307167174283
qrel_sabiazinho-4-2026-01-06_cot: 0.2881753150451688
qrel_sabiazinho-4-2026-01-06_simple: 0.29937434375693606


In [9]:
df_metricas_qrels_texto = metricas(df_resultados_texto, qrels, aproximacao_trec_eval=True, k=[5,10])
#display(df_metricas_qrels_texto.describe())
print('nDCG@10')
print(f'qrel humano: {df_metricas_qrels_texto['nDCG@10'].mean()}')
for nome_qrels, qrel_llm in qrels_llms.items():
    df_metricas_qrels_llm_texto = metricas(df_resultados_texto, qrel_llm, aproximacao_trec_eval=True, k=[5,10])
    print(f'{nome_qrels}: {df_metricas_qrels_llm_texto['nDCG@10'].mean()}')

nDCG@10
qrel humano: 0.3263065017451843
qrel_deepseek-chat_cot: 0.29889055328266173
qrel_deepseek-chat_simple: 0.31574583345819335
qrel_gpt-5-mini-2025-08-07_cot: 0.27404680811340176
qrel_gpt-5-mini-2025-08-07_simple: 0.28165897867819595
qrel_sabiazinho-4-2026-01-06_cot: 0.27351242034360973
qrel_sabiazinho-4-2026-01-06_simple: 0.2844090837080818


In [10]:
df_metricas_qrels_texto_e_assunto = metricas(df_resultados_texto_e_assunto, qrels, aproximacao_trec_eval=True, k=[5,10])
print('nDCG@10')
print(f'qrel humano: {df_metricas_qrels_texto_e_assunto['nDCG@10'].mean()}')
for nome_qrels, qrel_llm in qrels_llms.items():
    df_metricas_qrels_llm_texto_e_assunto = metricas(df_resultados_texto_e_assunto, qrel_llm, aproximacao_trec_eval=True, k=[5,10])
    print(f'{nome_qrels}: {df_metricas_qrels_llm_texto_e_assunto['nDCG@10'].mean()}')

nDCG@10
qrel humano: 0.35553884904198385
qrel_deepseek-chat_cot: 0.32234346915244827
qrel_deepseek-chat_simple: 0.3454135537192581
qrel_gpt-5-mini-2025-08-07_cot: 0.30104437796703754
qrel_gpt-5-mini-2025-08-07_simple: 0.30646956889266425
qrel_sabiazinho-4-2026-01-06_cot: 0.2993827131782839
qrel_sabiazinho-4-2026-01-06_simple: 0.31002292969412454
